# 📦 Northwind Traders Database Analysis

This notebook demonstrates Calliope AI using the **Northwind** database - a classic business operations sample database originally from Microsoft.

## 📊 Database Overview

Northwind Traders is a fictitious specialty foods import/export company. The database contains:
- **Products**: Product catalog with categories and suppliers
- **Orders**: Customer orders and order details
- **Customers**: Customer information and addresses
- **Employees**: Staff and their territories
- **Suppliers**: Product suppliers
- **Shippers**: Shipping companies

## 🔗 Real Datasource Connection

**Datasource ID**: `northwind`  
**Dialect**: PostgreSQL  
**Pre-configured** and ready to use!

## ⚙️ Configuration

**Run this cell first!** Sets up the Calliope API endpoint.

In [ ]:
import os

# Configure Calliope Data Agent API endpoint
# Change this if your setup is different
API_BASE_URL = 'http://localhost:5000'  # Local development
# API_BASE_URL = 'http://data-agent:5000'  # Docker/JupyterHub

os.environ['CALLIOPE_API_BASE_URL'] = API_BASE_URL
print(f"✅ Calliope API configured: {API_BASE_URL}")

## ✅ Verify Connection

In [ ]:
import requests

response = requests.get('http://localhost:5000/api/datasources')
datasources = response.json()['datasources']
northwind = [ds for ds in datasources if ds['id'] == 'northwind'][0]

print("✅ Connected to Northwind database!")
print(f"Datasource: {northwind['name']}")
print(f"Dialect: {northwind['dialect']}")

## 🔍 Explore Database Schema

In [ ]:
%%calliope run-sql northwind
SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'public' 
ORDER BY table_name

## 📦 Product Catalog Analysis

In [ ]:
%%calliope ask-sql northwind
How many products are in the catalog?

In [ ]:
%%calliope ask-sql northwind
Show product distribution by category

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
What are the top 10 most expensive products?

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Which products are discontinued?

## 💰 Sales Analysis

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
What is the total sales revenue?

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Show monthly sales trends

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
What are the top 10 best-selling products by revenue?

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Show revenue by product category

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Calculate year-over-year revenue growth

## 👥 Customer Analysis

In [ ]:
%%calliope ask-sql northwind
How many customers do we have?

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Show top 15 customers by total revenue

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Display customer distribution by country

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Calculate average order value per customer

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Find customers with no orders in the last 6 months

## 👔 Employee Performance

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Show sales performance by employee

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Which employee has the most orders?

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Calculate average order value by employee

## 📋 Order Analysis

In [ ]:
%%calliope ask-sql northwind
How many orders have been placed in total?

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
What is the average time between order date and ship date?

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Show order volume by country

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Calculate the average number of items per order

## 🏭 Supplier Analysis

In [ ]:
%%calliope ask-sql northwind
How many suppliers do we work with?

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Show supplier distribution by country

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Which suppliers provide the most products?

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Show total sales revenue generated by each supplier's products

## 🚚 Shipping Analysis

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Compare performance of different shipping companies by number of orders

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Calculate average freight cost by shipper

## 🌍 Geographic Analysis

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Show revenue by country (top 10)

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Which city has the most customers?

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Compare average order value by region

## 🚀 Advanced Analytics

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic --to-ai --model claude3
Analyze product profitability: identify which product categories have the best profit margins 
and recommend which categories we should focus on

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Perform customer segmentation: categorize customers as high, medium, or low value based on their total spending

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Calculate customer retention rate: what percentage of customers made repeat purchases?

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic --to-ai --model claude3
Identify seasonal trends in sales. Are there specific months or quarters with higher sales? 
What business strategies should we implement based on these patterns?

## 📊 Inventory & Stock Analysis

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Which products are low in stock (units in stock < reorder level)?

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Show products with high units on order

## 💻 Complex SQL Queries

In [ ]:
%%calliope run-sql northwind
SELECT 
    c.category_name,
    COUNT(DISTINCT p.product_id) AS product_count,
    COUNT(DISTINCT o.order_id) AS order_count,
    SUM(od.quantity) AS total_quantity_sold,
    ROUND(SUM(od.unit_price * od.quantity * (1 - od.discount))::numeric, 2) AS total_revenue,
    ROUND(AVG(p.unit_price)::numeric, 2) AS avg_product_price
FROM categories c
JOIN products p ON c.category_id = p.category_id
JOIN order_details od ON p.product_id = od.product_id
JOIN orders o ON od.order_id = o.order_id
GROUP BY c.category_id, c.category_name
ORDER BY total_revenue DESC

In [ ]:
%%calliope run-sql northwind
WITH customer_metrics AS (
    SELECT 
        c.customer_id,
        c.company_name,
        c.country,
        COUNT(DISTINCT o.order_id) AS order_count,
        SUM(od.unit_price * od.quantity * (1 - od.discount)) AS total_revenue,
        MIN(o.order_date) AS first_order_date,
        MAX(o.order_date) AS last_order_date
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_details od ON o.order_id = od.order_id
    GROUP BY c.customer_id, c.company_name, c.country
)
SELECT 
    company_name,
    country,
    order_count,
    ROUND(total_revenue::numeric, 2) AS total_revenue,
    ROUND((total_revenue / order_count)::numeric, 2) AS avg_order_value,
    first_order_date::date,
    last_order_date::date,
    (last_order_date::date - first_order_date::date) AS customer_lifespan_days
FROM customer_metrics
ORDER BY total_revenue DESC
LIMIT 20

## 🤔 Generate Business Insights

In [ ]:
%%calliope followup-questions northwind
What additional business questions should I ask about the Northwind Traders data?

## 📊 Key Insights

Using Calliope with Northwind database, we explored:
- Product catalog and inventory management
- Sales revenue analysis and trends
- Customer segmentation and behavior
- Employee performance metrics
- Supplier relationships
- Shipping and logistics
- Geographic market analysis
- Seasonal patterns and opportunities

## 🎯 Next Steps

Explore other real datasources:
- **Calliope-RealData-Sakila.ipynb** - DVD rental analytics
- **Calliope-RealData-Chinook.ipynb** - Music store analysis
- **Calliope-RealData-Employees.ipynb** - HR database